In [5]:
from sklearn.datasets import fetch_california_housing
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load the california housing dataset and split it into training and test sets.

california = fetch_california_housing(data_home="../../../scikit_learn_data", download_if_missing=False)
X = california.data
y = california.target


X = pd.DataFrame(X, columns=california.feature_names)
y = pd.Series(y, name="price")

dataset = pd.concat([X, y], axis=1)

# Remove the Latitude and Longitude columns from the dataset
dataset = dataset.drop(columns=["Latitude", "Longitude"])

In [ ]:
# Create a pairplot to visualize the relationships between features and the target variable
# Currently disabled to avoid long execution time and cluttered output. Uncomment the line below to enable it.
#sns.pairplot(dataset, hue="price", palette="viridis")

In [ ]:
# Create a pairplot to visualize the relationships between features and the target variable
# Currently disabled to avoid long execution time and cluttered output. Uncomment the line below to enable it.
#sns.pairplot(dataset, vars=['HouseAge', 'MedInc', 'Population'], hue='price', palette='viridis')

In [ ]:
dataset = dataset[dataset['price'] <= 4.999]  # Filter out rows where price is greater than or equal to 5

In [ ]:
#sns.pairplot(dataset)

In [ ]:
from sklearn.preprocessing import StandardScaler
# We should scale the features before training the model. We will use StandardScaler to scale the features to have zero mean and unit variance.

scaler = StandardScaler()
dataset_scaled = scaler.fit_transform(dataset)
dataset = pd.DataFrame(dataset_scaled, columns=dataset.columns)

In [ ]:
import numpy as np
from sklearn.linear_model import ElasticNet
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

X = dataset.drop(columns=['price'])
y = dataset['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#alphas_tmp = range(-30, 10, 1)
alphas_tmp = range(-60, 0, 5)
alphas = [10 ** (i/20) for i in alphas_tmp]

l1_ratios_tmp = range(10, 100, 10)
l1_ratios = [i/100 for i in l1_ratios_tmp]

output = pd.DataFrame(columns=['alpha', 'l1_ratio', 'r2_score'])
for alpha in alphas:
    for l1_ratio in l1_ratios:
        elasticnet_model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio).fit(X_train, y_train)
        # Add the results to the output DataFrame
        r2_score = elasticnet_model.score(X_test, y_test)
        output = pd.concat([output, pd.DataFrame({'alpha': [alpha], 'l1_ratio': [l1_ratio], 'r2_score': [r2_score]})], ignore_index=True)

# Create a 3D surface plot of the R2 scores as a function of alpha and l1_ratio
fig = go.Figure(data=[go.Surface(z=output.pivot(index='l1_ratio', columns='alpha', values='r2_score').values,
                         x=output['alpha'].unique(),
                         y=output['l1_ratio'].unique())])
fig.update_layout(title='R2 Score vs Alpha and L1', autosize=False,
                  width=800, height=800,
                  margin=dict(l=65, r=50, b=65, t=90))
fig.update_layout(scene=dict(
                    xaxis=dict(type='log'),
                    xaxis_title='Alpha',
                    yaxis_title='L1 Ratio',
                    zaxis_title='R2 Score'),
                    )
fig.show()

